# AIG-KG demo — neurogenetic (TTR/ATTR) knowledge graph

This notebook **orchestrates and visualizes**; the logic lives in the importable `aig_kg`
package. It runs on synthetic data by default (no PHI); swap in real EMR by pointing
`ingest.load_all` at a directory of your `*_sample.tsv`-shaped extracts.

Pipeline: **generate/ingest → build_kg (NetworkX) → U1 delay · U2 density · U3 comorbidity · U4 CIDP screen → explore**.

In [ ]:
import pandas as pd
from aig_kg import synth, ingest
from aig_kg.graph import build_kg
from aig_kg.analytics import (diagnostic_delay, encounter_density,
                              comorbidity_and_control, screen_cidp_mimics)
from aig_kg.analytics import viz

# --- data source: synthetic by default -------------------------------------------------
person, events = synth.generate(seed=42)
# To use your real EMR extracts instead, comment the line above and use:
# person, events = ingest.load_all('../tests/fixtures')   # dir of *_sample.tsv files

g = build_kg(person, events)
print(f'{g.number_of_nodes()} nodes, {g.number_of_edges()} edges, '
      f'{len(person)} patients, {len(events)} events')

## U1 — diagnostic delay (first red-flag → ATTR diagnosis / therapy)

In [ ]:
delay = diagnostic_delay(g)
display(delay[['person_id','phenotype','first_feature','t_dx','delay_days_dx','delay_days_therapy']])
print('median dx delay (yrs):', round(pd.to_numeric(delay['delay_days_dx']).median()/365.25, 1))

# quick plot (optional; needs matplotlib)
try:
    ax = pd.to_numeric(delay['delay_days_dx']).div(365.25).plot(kind='hist', bins=8,
         title='ATTR diagnostic delay (years)')
    ax.set_xlabel('years from first feature to ATTR diagnosis')
except Exception as e:
    print('plot skipped:', e)

## U2 — encounter density / care burden (in-person vs virtual)

In [ ]:
density = encounter_density(g)
display(density.sort_values('contacts_per_year', ascending=False).head(10))

## U3 — comorbidity & control (diabetes + HbA1c trajectory)

In [ ]:
cc = comorbidity_and_control(g)
dm = cc[cc['has_diabetes']]
display(dm[['person_id','n_hba1c','first_hba1c','last_hba1c']].head(10))

# one patient's HbA1c trajectory
try:
    import matplotlib.pyplot as plt
    row = dm[dm['n_hba1c'] > 2].iloc[0]
    traj = row['hba1c_trajectory']
    xs, ys = [d for d, _ in traj], [v for _, v in traj]
    plt.plot(xs, ys, marker='o'); plt.title(f"HbA1c — {row['person_id']}")
    plt.ylabel('HbA1c (%)'); plt.show()
except Exception as e:
    print('plot skipped:', e)

## U4 — CIDP vs mimic screen

Flags patients *labeled* CIDP who carry EMR-derivable mimic red-flags (autonomic involvement,
muscle atrophy, ATTR features) for genetic/amyloid workup. `cidp_prob_partial` uses only the
EMR-available calculator inputs — see `analytics/cidp.py` on intercept calibration.

In [ ]:
screen = screen_cidp_mimics(g)
display(screen)
print('flagged for workup:', int(screen['recommend_genetic_workup'].sum()), 'of', len(screen))

## Explore — one patient's trajectory + the KG subgraph

In [ ]:
pid = delay['person_id'].iloc[0]   # an ATTR patient
display(viz.patient_timeline_df(g, pid))

# interactive graph (needs the 'viz' extra: pip install 'aig-kg[viz]')
try:
    net = viz.to_pyvis(g, pid)
    net.show(f'patient_{pid}.html')
except Exception as e:
    print('pyvis skipped:', e)